# 🚀 ITIR Remote Prover Runner
### High-RAM (52 GB) Lean 4 & Agda Cloud Execution Lane

This notebook connects your Google Colab High-RAM runtime to the **ITIR Prover Relay**.
It claims queued exact-SHA compilation jobs from the `colab-runner-state` branch in `chboishabba/ITIR-suite`, executes them in isolated worktrees, and posts verified execution receipts and logs back to GitHub.

---

### Step 1: Verify Hardware & Inject GitHub Token
Make sure you have added `ITIR_COLAB_GITHUB_TOKEN` in the **Secrets** (key icon) on the left sidebar with notebook access enabled.

In [ ]:
import os
import psutil
from google.colab import userdata

ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"Detected System RAM: {ram_gb:.2f} GB")
if ram_gb > 25:
    print("✅ High-RAM environment active!")
else:
    print("ℹ️ Note: To use the 52 GB allocation, go to Runtime -> Change runtime type -> High-RAM.")

try:
    token = userdata.get("ITIR_COLAB_GITHUB_TOKEN")
    if token:
        os.environ["ITIR_COLAB_GITHUB_TOKEN"] = token
        print("✅ ITIR_COLAB_GITHUB_TOKEN successfully loaded from Colab Secrets.")
    else:
        print("⚠️ ITIR_COLAB_GITHUB_TOKEN is empty in Colab Secrets.")
except Exception as e:
    print(f"⚠️ Error reading secret: {e}")
    print("Please add ITIR_COLAB_GITHUB_TOKEN in the Secrets tab (key icon) on the left sidebar.")

### Step 2: Bootstrap Toolchains
Clones `ITIR-suite` and installs `elan`, Lean `v4.35.0-rc3`, matching `lake`, and Agda.

In [ ]:
%%bash
set -e
git clone https://github.com/chboishabba/ITIR-suite.git /content/ITIR-suite 2>/dev/null || (cd /content/ITIR-suite && git fetch origin)
cd /content/ITIR-suite
git checkout main
git pull --ff-only origin main

python scripts/colab_bootstrap_provers.py

### Step 3: Run the Prover Worker
Polls the `colab-runner-state` queue for Lean 4 and Agda jobs and streams results.

In [ ]:
%%bash
cd /content/ITIR-suite
python scripts/colab_worker.py --poll